# Afina tu propio LLM

**seLIA 2026 · URJC Fuenlabrada** · Asociación de IA (URJC) · OfiLibre

Afinamos (fine-tuning) un **modelo de lenguaje abierto con tus propios datos**, en local y con software libre, usando **LoRA** sobre el stack estándar de Hugging Face (`transformers` + `peft`).

Pipeline: `Cargar base` → `Añadir LoRA` → `Entrenar` → `Probar` → `Guardar`.

> Este notebook está pensado para ejecutarse **con solo `git` y `pip`** (sin conda ni permisos de administrador) sobre una **GPU potente**. Sigue primero el `README.md`. Publicado bajo **CC BY-SA 4.0**.

## ⚠️ Antes de nada (Google Colab)

1. **Activa GPU**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `T4 GPU` (o superior).
2. Ejecuta la celda siguiente para instalar las dependencias necesarias (Colab ya trae PyTorch con CUDA, así que **no** lo reinstalamos).
3. El resto del notebook es idéntico al del taller original.


In [1]:
# Instalación de dependencias (Colab ya trae torch con CUDA preinstalado)
!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.4 MB/s eta 0:00:00


## 0 - Comprobar la GPU
Si no nos salta `True`, revisamos `README.md` (instalación de PyTorch con CUDA).

In [2]:
import torch
print("PyTorch:", torch.__version__)
print("¿CUDA disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bfloat16 soportado:", torch.cuda.is_bf16_supported())

PyTorch: 2.11.0+cu128
¿CUDA disponible?: True
GPU: Tesla T4
bfloat16 soportado: True


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detecta GPU. Ve a 'Entorno de ejecución' -> 'Cambiar tipo de entorno de ejecución' "
        "y selecciona una GPU (T4 gratuita o superior), luego vuelve a ejecutar el notebook."
    )


## 1 - Configuración
Con una GPU potente vamos en **16 bits** (sin cuantización).
Si la GPU usada para las pruebas tiene poca memoria, pon `LOAD_IN_4BIT = True` (necesita `bitsandbytes`, ver README).

In [3]:
MODELO   = "Qwen/Qwen2.5-1.5B-Instruct"   # abierto (Apache-2.0), sin gate. Prueba 3B/7B si te sobra VRAM.
MAX_LEN  = 1024
LOAD_IN_4BIT = False                       # True = QLoRA (menos memoria, requiere bitsandbytes)
SALIDA   = "afin_lora"

DEV = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

## 2 - Cargar la base
Descargamos el modelo y su tokenizador. La primera vez tarda un poco (lo bajamos desde Hugging Face).

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map={"": 0})
else:
    model = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=DTYPE).to(DEV)

print("Parámetros (millones):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Parámetros (millones): 1543.7


### 2.1 - ¿Cómo responde nuestro modelo de lenguaje antes de afinar?
El modelo base responde de forma genérica ateniéndose a cómo lo entrenaron originalmente.

In [9]:
import torch

def responde(mensaje, max_new_tokens=120):
    model.eval()
    msgs = [{"role": "user", "content": mensaje}]

    # Generamos los inputs
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    if hasattr(inputs, "keys") or isinstance(inputs, dict):
        input_ids = inputs["input_ids"].to(model.device)
        generacion_kwargs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        input_ids = inputs.to(model.device)
        generacion_kwargs = {"input_ids": input_ids}

    prompt_length = input_ids.shape[1]

    with torch.no_grad():
        out = model.generate(
            **generacion_kwargs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    texto = tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True)
    print(texto)
    return texto

# Prueba la función
_ = responde("¿Quién eres y quién te ha creado?")

Como asistente de inteligencia artificial desarrollado por Alibaba Cloud, mi nombre es Qwen. Fui creado para ayudar a las personas a obtener información y resolver problemas utilizando la tecnología de IA. Mi objetivo es proporcionar respuestas precisas y útiles basadas en los datos disponibles. No tengo una identidad personal ni un creador humano específico, ya que soy un programa de código escrito por expertos en el campo de la IA. Mi propósito principal es facilitar la comunicación entre usuarios y ofrecerles asistencia informática y de otras áreas relevantes.


## 3 - Añadimos los adaptadores LoRA
Tomamos el LLM y lo paramos para poder entrenar solo matrices pequeñas (**LoRA**, es decir, no la red completa): menos del 1 % de los pesos.

In [11]:
!pip install --upgrade "torchao>=0.16.0"

from peft import LoraConfig, get_peft_model

if LOAD_IN_4BIT:
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4 - Tus datos mandan
El fine-tuning aprende de pares **{instrucción → respuesta}**. Este mini-dataset le da una **personalidad concreta**. Para este caso, será el asistente basado en IA Libre de la Asociación de Inteligencia Artificial. Con pocos ejemplos ya podemos notar el cambio.

**ACTIVIDAD**: sustituye estos pares por los tuyos. La regla de oro: *calidad > cantidad*.

In [12]:
EJEMPLOS = [
    ("¿Quién eres?",
     "Soy el asistente basado en IA Libre de la Asociación de IA de la URJC. Te ayudo con IA usando siempre software libre."),
    ("¿Quién te ha creado?",
     "Me han afinado en el taller de seLIA con LoRA, con datos abiertos y software libre."),
    ("¿Qué es la IA libre?",
     "Es la IA con código y pesos abiertos: puedes ver cómo funciona, ejecutarla en tu máquina y modificarla."),
    ("¿Por qué usar modelos abiertos?",
     "Por privacidad, independencia y coste: tus datos no salen de tu equipo y no pagas suscripciones."),
    ("¿Qué es el fine-tuning?",
     "Es ajustar los pesos de un modelo con tus ejemplos para fijar su estilo o su tarea."),
    ("Diferencia entre LoRA y QLoRA.",
     "LoRA entrena adaptadores pequeños con la base congelada; QLoRA además cuantiza la base a 4 bits para gastar mucha menos memoria."),
    ("¿Qué es RAG?",
     "Es recuperar documentos y añadirlos al prompt para dar contexto fresco sin tocar los pesos."),
    ("¿Cuándo conviene afinar en vez de solo hacer prompting?",
     "Cuando necesitas un comportamiento estable que el prompt por sí solo no consigue."),
    ("Recomiéndame una herramienta para chatear con LLMs en local.",
     "Prueba Jan.ai o GPT4All: funcionan en local, incluso sin GPU potente."),
    ("¿Qué licencia debería poner a mi dataset?",
     "Si quieres que otros lo reutilicen y compartan igual, una Creative Commons como CC BY-SA encaja bien."),
    ("Dame un consejo para preparar datos de entrenamiento.",
     "Pocos ejemplos buenos y variados superan a miles ruidosos, y respeta la plantilla de chat del modelo."),
    ("¿Qué es Unsloth?",
     "Una librería libre que acelera el fine-tuning y ahorra memoria."),
    ("Explícame los embeddings en una frase.",
     "Son vectores que representan el significado de un texto para poder compararlo."),
    ("¿Qué GPU necesito para afinar un modelo de 7B?",
     "Con LoRA en 16 bits te vale una GPU de gama alta; con QLoRA bajas a unos 8–16 GB."),
    ("Salúdame.",
     "¡Hola! Estaré encantado de ayudarte con IA libre."),
    ("¿Qué es un token en un LLM?",
     "Es la unidad mínima de texto que procesa el modelo, normalmente un trozo de palabra."),
    ("¿Qué significa que un modelo alucine?",
     "Que genera información plausible pero falsa; conviene verificar datos importantes."),
    ("¿Puedo ejecutar mi modelo afinado sin conexión?",
     "Sí: fusiona los adaptadores, expórtalo a GGUF y córrelo con Ollama."),
    ("¿Qué es Hugging Face?",
     "Una plataforma con modelos, datasets y librerías abiertas como transformers y PEFT."),
    ("Motívame para empezar en la IA.",
     "Con un portátil y curiosidad ya puedes afinar tu primer modelo hoy mismo."),
    ("¿Qué es OfiLibre?",
     "La oficina de tecnologías libres de la URJC, que promueve software y ciencia abiertos."),
    ("¿Cómo comparto mi modelo afinado?",
     "Súbelo a Hugging Face con su licencia y documenta cómo lo entrenaste."),
    ("¿Qué es SFT?",
     "Supervised Fine-Tuning: afinar con pares de instrucción y respuesta."),
    ("Despídete.",
     "¡Hasta pronto! Sigue afinando y compartiendo en abierto."),
]
print(len(EJEMPLOS), "ejemplos")

24 ejemplos


In [13]:
from datasets import Dataset

def a_texto(par):
    user, assistant = par
    msgs = [{"role": "user", "content": user},
            {"role": "assistant", "content": assistant}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

def tokeniza(lote):
    out = tokenizer(lote["text"], truncation=True, max_length=MAX_LEN)
    return out

dataset = Dataset.from_dict({"text": [a_texto(p) for p in EJEMPLOS]})
dataset = dataset.map(tokeniza, remove_columns=["text"])
print(dataset)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 24
})


## 5 - Entrenar
Dataset diminuto → basta con **3 épocas** (unos segundos en una GPU potente). Sube `num_train_epochs` para entrenar más.

En caso de tener una GPU poco potente, además de haber cambiado al framework de QLoRA, es recomendable cambiar las siguientes variables para que sea más manejable la carga:
- per_device_train_batch_size = 1
- gradient_accumulation_steps = 1
- num_train_epochs = 12

Recomendamos experimentar con estas variables, es buena práctica el consultar la librería: https://huggingface.co/docs/transformers/v5.13.0/en/main_classes/trainer#transformers.TrainingArguments

In [14]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=12,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=5,
    bf16=(DTYPE == torch.bfloat16),
    fp16=(DTYPE == torch.float16),
    lr_scheduler_type="linear",
    optim="adamw_torch",
    save_strategy="no",
    report_to="none",
    seed=3407,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
5,4.505830
10,3.115306
15,2.003510
20,1.710291
25,1.408787
30,1.198542
35,1.094269
40,0.787450
45,0.763076
50,0.700063


TrainOutput(global_step=144, training_loss=0.6896204594522715, metrics={'train_runtime': 158.0292, 'train_samples_per_second': 1.822, 'train_steps_per_second': 0.911, 'total_flos': 144754742439936.0, 'train_loss': 0.6896204594522715, 'epoch': 12.0})

## 6 - Probamos después del afinado
Repetimos las preguntas del paso 2.1. Deberíamos ver la nueva personalidad del modelo.

In [15]:
_ = responde("¿Quién eres y quién te ha creado?")
print("\n" + "="*60 + "\n")
_ = responde("¿Qué es la IA libre y por qué usar modelos abiertos?")

Soy el asistente basado en IA Libre de la Asociación de IA de la URJC. Te ayudo con IA usando siempre software libre.


Es la IA con código y pesos abiertos; conviene que también el licencier desposee derechos sobre los datos.


### 💾 (Opcional) Guardar en Google Drive

En Colab, todo lo que guardes en `/content` **se borra al cerrar la sesión**. Si quieres conservar los adaptadores o el modelo fusionado, monta tu Drive y cambia `SALIDA` para que apunte ahí (por ejemplo `"/content/drive/MyDrive/afin_lora"`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Descomenta y ajusta si quieres guardar en Drive en vez de en /content:
# SALIDA = "/content/drive/MyDrive/afin_lora"


## 7 - Guardar
Guardamos los **adaptadores LoRA** (ligeros). Los podemos **fusionar** con el modelo base para tener uno autónomo.

In [ ]:
# a) Solo adaptadores (pequeños)
model.save_pretrained(SALIDA)
tokenizer.save_pretrained(SALIDA)
print("Adaptadores LoRA guardados en ./" + SALIDA)

Adaptadores LoRA guardados en ./afin_lora


In [ ]:
# b) (opcional) Fusionar adaptadores + base -> modelo completo en 16 bits
#    Requiere no estar en 4-bit. Útil para exportar a GGUF/Ollama después.
if not LOAD_IN_4BIT:
    merged = model.merge_and_unload()
    merged.save_pretrained("afin_merged")
    tokenizer.save_pretrained("afin_merged")
    print("Modelo fusionado en ./afin_merged")
else:
    print("En 4-bit no se fusiona; reentrena con LOAD_IN_4BIT=False si necesitas fusionar.")

En 4-bit no se fusiona; reentrena con LOAD_IN_4BIT=False si necesitas fusionar.


## 8 - Información extra - Ejecutarlo con Ollama, sin conexión
Solo con **git + pip**. Convertimos el modelo fusionado a **GGUF** con `llama.cpp` y así podemos correrlo en nuestra máquina con [Ollama](https://ollama.com) por ejemplo.

```bash
# 1) Clonar y preparar llama.cpp (solo git + pip)
git clone https://github.com/ggerganov/llama.cpp
pip install -r llama.cpp/requirements.txt

# 2) Convertir a GGUF (cuantizado q4_k_m)
python llama.cpp/convert_hf_to_gguf.py afin_merged --outfile afin.gguf --outtype q8_0

# 3) Modelfile de Ollama
printf 'FROM ./afin.gguf\nPARAMETER temperature 0.7\nSYSTEM "Eres Afin, el asistente de IA Libre de la URJC."\n' > Modelfile
ollama create afin -f Modelfile
ollama run afin
```


## 9 - ¡A experimentar!
1. **Cambia los datos** (paso 4) por tu dominio o estilo.
2. **Cambia el modelo** (paso 1): `Qwen/Qwen2.5-3B-Instruct`, `meta-llama/Llama-3.2-3B-Instruct`, `google/gemma-2-2b-it`… prueba los que quieras!
3. **Ajusta el entrenamiento** (paso 5): `num_train_epochs`, `r` del LoRA, `learning_rate`.
4. Vuelve a **probar** (paso 6) y compara.